<a href="https://colab.research.google.com/github/krmasif/Agent/blob/main/training_a_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip -q install transformers datasets accelerate evaluate

In [2]:
from huggingface_hub import notebook_login
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
!apt -y install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [4]:
!nvidia-smi

Sat Mar 28 07:24:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=5
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
def test_raw_distilbert():
    print("🧪 Testing raw distilbert-base-uncased (no classification head)...")

    from transformers import AutoModel, AutoTokenizer
    import torch # Import torch here

    # Load the raw base model
    base_model = AutoModel.from_pretrained("distilbert-base-uncased")
    base_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

    test_phrases = ["I love this!", "This is terrible"]

    for phrase in test_phrases:
        inputs = base_tokenizer(phrase, return_tensors="pt", truncation=True)

        with torch.no_grad():
            outputs = base_model(**inputs)
            # outputs.last_hidden_state contains embeddings, not classifications

        print(f"'{phrase}' → Raw embeddings shape: {outputs.last_hidden_state.shape}")
        print("Note: Base model doesn't classify, it creates embeddings!")

In [20]:
test_raw_distilbert()

🧪 Testing raw distilbert-base-uncased (no classification head)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'I love this!' → Raw embeddings shape: torch.Size([1, 6, 768])
Note: Base model doesn't classify, it creates embeddings!
'This is terrible' → Raw embeddings shape: torch.Size([1, 5, 768])
Note: Base model doesn't classify, it creates embeddings!


In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

encoded = dataset.map(tokenize_function, batched=True)
encoded = encoded.remove_columns(["text"])
encoded = encoded.rename_column("label", "labels")
# Remove or comment out the following line:
# encoded.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="ft-distilbert-yelp",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,  # good on most Colab GPUs; if errors, set fp16=False
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"].shuffle(seed=42).select(range(20000)),
    eval_dataset=encoded["test"].select(range(2000)),
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.853723,0.947947,0.568500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.9959860778808594, metrics={'train_runtime': 239.7243, 'train_samples_per_second': 83.429, 'train_steps_per_second': 5.214, 'total_flos': 2353591391944320.0, 'train_loss': 0.9959860778808594, 'epoch': 1.0})

In [10]:
trainer.evaluate()

{'eval_loss': 0.9479474425315857,
 'eval_accuracy': 0.5685,
 'eval_runtime': 11.1758,
 'eval_samples_per_second': 178.958,
 'eval_steps_per_second': 11.185,
 'epoch': 1.0}

In [14]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

In [15]:
trainer.state.log_history

[{'loss': 1.547152099609375,
  'grad_norm': 2.844700336456299,
  'learning_rate': 1.9216e-05,
  'epoch': 0.04,
  'step': 50},
 {'loss': 1.2471701049804687,
  'grad_norm': 5.367774486541748,
  'learning_rate': 1.8416e-05,
  'epoch': 0.08,
  'step': 100},
 {'loss': 1.1239193725585936,
  'grad_norm': 5.7496209144592285,
  'learning_rate': 1.7616000000000002e-05,
  'epoch': 0.12,
  'step': 150},
 {'loss': 1.0456588745117188,
  'grad_norm': 4.57061767578125,
  'learning_rate': 1.6816e-05,
  'epoch': 0.16,
  'step': 200},
 {'loss': 1.0750624084472655,
  'grad_norm': 11.456097602844238,
  'learning_rate': 1.6016e-05,
  'epoch': 0.2,
  'step': 250},
 {'loss': 1.021324920654297,
  'grad_norm': 7.262930393218994,
  'learning_rate': 1.5216000000000001e-05,
  'epoch': 0.24,
  'step': 300},
 {'loss': 0.9823098754882813,
  'grad_norm': 9.251636505126953,
  'learning_rate': 1.4416e-05,
  'epoch': 0.28,
  'step': 350},
 {'loss': 0.9656471252441406,
  'grad_norm': 12.844730377197266,
  'learning_rate':

In [16]:
import pandas as pd
pd.DataFrame(trainer.state.log_history)

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_accuracy,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,1.547152,2.844700,1.921600e-05,0.04,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.247170,5.367774,1.841600e-05,0.08,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.123919,5.749621,1.761600e-05,0.12,150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.045659,4.570618,1.681600e-05,0.16,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.075062,11.456098,1.601600e-05,0.20,250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1.021325,7.262930,1.521600e-05,0.24,300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.982310,9.251637,1.441600e-05,0.28,350,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.965647,12.844730,1.361600e-05,0.32,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.979437,8.318737,1.281600e-05,0.36,450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0.963251,8.783406,1.201600e-05,0.40,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
predictions = trainer.predict(encoded["test"])
logits = predictions.predictions
labels = predictions.label_ids

preds = np.argmax(logits, axis=-1)

In [18]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(labels, preds))
print(classification_report(labels, preds))

[[7554 2151  174   53   68]
 [2308 5874 1556  201   61]
 [ 333 2645 4712 2066  244]
 [  89  284 1918 5308 2401]
 [ 107   73  337 2621 6862]]
              precision    recall  f1-score   support

           0       0.73      0.76      0.74     10000
           1       0.53      0.59      0.56     10000
           2       0.54      0.47      0.50     10000
           3       0.52      0.53      0.52     10000
           4       0.71      0.69      0.70     10000

    accuracy                           0.61     50000
   macro avg       0.61      0.61      0.61     50000
weighted avg       0.61      0.61      0.61     50000



In [30]:
def quick_training_check():
    """Test your fine-tuning progress safely"""
    print("🔍 Quick check on current training progress...")

    import torch # Import torch here

    test_phrases = [
        "I love this product!",
        "This is terrible",
        "Average experience",
        "Absolutely wonderful!",
        "Completely disappointed",
        "Trainer is not good"
    ]

    # Ensure model is in eval mode
    model.eval()

    with torch.no_grad():
        for phrase in test_phrases:
            inputs = tokenizer(phrase, return_tensors="pt", truncation=True, padding=True)

            # Move inputs to the same device as the model
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            # Get predictions from your fine-tuned model
            outputs = model(**inputs)

            # Handle different output formats
            if isinstance(outputs, dict) and 'logits' in outputs:
                logits = outputs['logits']
            elif hasattr(outputs, 'logits'):
                logits = outputs.logits
            else:
                logits = outputs[0]

            # Convert to probabilities and predictions
            probs = torch.nn.functional.softmax(logits, dim=-1)
            pred = torch.argmax(probs, dim=-1)

            # Convert to labels for a 5-class classification (Yelp dataset)
            label_map = {0: "1-star (Negative)", 1: "2-star (Negative)", 2: "3-star (Neutral)", 3: "4-star (Positive)", 4: "5-star (Positive)"}
            sentiment = label_map.get(pred.item(), "Unknown")
            confidence = probs[0][pred].item()

            print(f"'{phrase}' → {sentiment} ({confidence:.3f})")

    # Set model back to train mode for continued training
    model.train()

In [31]:
quick_training_check()

🔍 Quick check on current training progress...
'I love this product!' → 5-star (Positive) (0.864)
'This is terrible' → 1-star (Negative) (0.877)
'Average experience' → 2-star (Negative) (0.438)
'Absolutely wonderful!' → 5-star (Positive) (0.879)
'Completely disappointed' → 1-star (Negative) (0.753)
'Trainer is not good' → 1-star (Negative) (0.586)
